# 문제 1-1 : 기본 Chain 만들기 - AI 요리사

사용자가 재료를 입력하면 그 재료로 만들 수 있는 요리와 간단한 레시피를 추천합니다.

- `PromptTemplate`으로 프롬프트 작성
- `ChatOpenAI` 모델 사용
- `StrOutputParser`로 결과를 문자열로 출력
- LCEL(`|`) 문법으로 체인 연결

## 1. 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
print("OpenAI Key:", "OK" if os.getenv("OPENAI_API_KEY") else "없음")

OpenAI Key: OK


## 2. 구성 요소 만들기 (Prompt / Model / Parser)

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 1) 프롬프트: {ingredients} 자리에 사용자가 입력한 재료가 들어감
template_text = """당신은 친절한 AI 요리사입니다.
사용자가 가진 재료로 만들 수 있는 요리를 1가지 추천하고 간단한 레시피를 알려주세요.

재료: {ingredients}

아래 형식으로 한국어로 답변하세요.

{ingredients}(으)로 만들 수 있는 요리를 추천드립니다!

추천 요리: (요리 이름)
재료: (필요한 재료)
조리법:
1. ...
2. ...
3. ...
"""
prompt = PromptTemplate.from_template(template_text)

# 2) 모델
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 3) 출력 파서: AIMessage -> str
output_parser = StrOutputParser()

## 3. LCEL로 체인 연결

In [3]:
chain = prompt | llm | output_parser
print(type(chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


## 4. 실행

In [4]:
result = chain.invoke({"ingredients": "계란, 밥, 김치"})
print(isinstance(result, str))  # True : 문자열로 변환됨
print(result)

True
계란, 밥, 김치(으)로 만들 수 있는 요리를 추천드립니다!

추천 요리: 김치 볶음밥
재료: 밥 1공기, 계란 1개, 김치 1컵, 식용유, 소금, 후추, 대파 (선택사항)

조리법:
1. 팬에 식용유를 두르고 중불에서 잘게 썬 김치를 넣고 볶아줍니다.
2. 김치가 익으면 밥을 넣고 함께 볶아줍니다. 이때 소금과 후추로 간을 맞춥니다.
3. 다른 팬에서 계란을 프라이 해줍니다. 볶음밥 위에 계란 프라이를 올리고 대파를 뿌리면 완성입니다. 

맛있게 드세요!


In [5]:
# 다른 재료로도 테스트
result = chain.invoke({"ingredients": "토마토, 양파, 치즈"})
print(result)

토마토, 양파, 치즈(으)로 만들 수 있는 요리를 추천드립니다!

추천 요리: 토마토 양파 치즈 샐러드  
재료:  
- 토마토 2개  
- 양파 1개  
- 치즈 (모짜렐라 또는 체다) 적당량  
- 올리브유 2큰술  
- 소금, 후추 적당량  
- 바질 (선택 사항)  

조리법:  
1. 토마토와 양파를 먹기 좋은 크기로 썰어줍니다.  
2. 썰어놓은 토마토와 양파를 큰 볼에 담고, 치즈를 적당한 크기로 잘라 넣습니다.  
3. 올리브유, 소금, 후추를 넣고 가볍게 섞어준 후, 원한다면 바질을 추가하여 장식합니다.  

맛있게 드세요!
